# 02 — Parameter Sweep (the misleading heatmap)

This notebook demonstrates the data-snooping problem.

We sweep a 20×20 grid of (fast, slow) windows on SPY in-sample, plot the Sharpe heatmap, and pick the 'best' parameters. Then we use the Deflated Sharpe Ratio to show that the best Sharpe is, in expectation, what you'd get by sweeping 400 strategies with zero true skill.

**This heatmap is what most amateur backtests report as their headline. It is the wrong number.**

In [ ]:
import pandas as pd
import numpy as np

from ma_backtester.backtester import sweep as run_sweep
from ma_backtester.config import DEFAULT_SWEEP, CostConfig, StrategyConfig
from ma_backtester.costs import FixedBpsCost
from ma_backtester.data import load_close
from ma_backtester.data_snooping import deflated_sharpe_ratio, effective_number_of_trials
from ma_backtester.metrics import sharpe_ratio
from ma_backtester.plotting import install_theme, parameter_heatmap

install_theme()
close = load_close('SPY', start='2010-01-01', end='2024-12-31')
cost = FixedBpsCost(CostConfig(per_side_bps=5.0))

## Run the sweep

In [ ]:
grid = DEFAULT_SWEEP.grid()
print(f'Running {len(grid)} configurations...')
results = run_sweep(close=close, grid=grid, cost_model=cost)
sharpes = {cfg: sharpe_ratio(res.daily_returns) for cfg, res in results.items()}
best_cfg = max(sharpes, key=lambda c: sharpes[c])
print(f'Best in-sample: SMA({best_cfg.fast_window}, {best_cfg.slow_window})  Sharpe = {sharpes[best_cfg]:.3f}')

## Heatmap — diagnostic only

The colour does not represent expected future performance. It represents how well each pair *happened to fit one historical path*.

In [ ]:
grid_df = pd.DataFrame(
    index=sorted(DEFAULT_SWEEP.fast_windows),
    columns=sorted(DEFAULT_SWEEP.slow_windows),
    dtype='float64',
)
for cfg, s in sharpes.items():
    grid_df.loc[cfg.fast_window, cfg.slow_window] = s

parameter_heatmap(sharpe_grid=grid_df)

## Effective number of trials

The 400 strategies are highly correlated — most pairs produce similar signals. We use PCA on the strategy-return matrix to estimate the effective number of *independent* trials. This is what the Deflated Sharpe Ratio should use, not the raw 400.

In [ ]:
returns_matrix = pd.DataFrame({
    f'{cfg.fast_window}_{cfg.slow_window}': res.daily_returns
    for cfg, res in results.items()
})
n_eff = effective_number_of_trials(returns_matrix=returns_matrix)
print(f'Effective number of independent trials (PCA, 95% var): {n_eff} of {len(grid)}')

## Deflated Sharpe Ratio

Bailey & López de Prado (2014). Adjusts the observed Sharpe for selection bias across trials, non-normality of returns, and sample length. Returns the probability that the *true* annualised Sharpe is positive after accounting for how hard we searched.

Convention: reject the null of no skill only if DSR > 0.95.

In [ ]:
dsr = deflated_sharpe_ratio(
    daily_returns=results[best_cfg].daily_returns,
    n_trials=len(grid),
    n_effective_trials=n_eff,
)
from dataclasses import asdict
pd.Series(asdict(dsr))

## The honest take

Compare `observed_sharpe` to `expected_max_sharpe_under_null`. The latter is what 400 strategies with zero skill would produce *just by luck*. If the observed Sharpe doesn't materially exceed that, we have learned nothing from the sweep.